# YOLOv8 Coral Detection Model Training

This notebook trains a YOLOv8 model using your 341 augmented coral images from Roboflow.

**Dataset:** 341 augmented images (110 base × 4 augmentation)

**7 Coral Classes:** Hard Coral, Soft Coral, Macroalgae, Halimeda, Algae Assemblage, Abiotic, Other Biota

## Step 1: Install Dependencies

In [ ]:
# Install YOLOv8 and dependencies
!pip install ultralytics roboflow -q
print("✓ Dependencies installed successfully!")

## Step 2: Download Dataset from Roboflow

**PASTE YOUR ROBOFLOW CODE HERE:**

Replace the code in the next cell with the download code from Roboflow (the code you just copied)

In [ ]:
# PASTE YOUR ROBOFLOW DOWNLOAD CODE HERE
from roboflow import Roboflow

rf = Roboflow(api_key="zdPSV7Poje2dGgQVyLi8")
project = rf.workspace("christiandominiques-workspace").project("coral-detection-e21y1")
dataset = project.versions(1).download("yolov8")

print(f"✓ Dataset downloaded to: {dataset.location}")

## Step 3: Verify Dataset Structure

In [ ]:
import os
import yaml

# Check dataset structure
dataset_path = dataset.location
print(f"Dataset path: {dataset_path}\n")

# List contents
print("Dataset contents:")
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        count = len(os.listdir(item_path))
        print(f"  📁 {item}/: {count} items")
    else:
        print(f"  📄 {item}")

# Load and display data.yaml
yaml_path = os.path.join(dataset_path, 'data.yaml')
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)
    print(f"\n✓ Classes found: {len(data_yaml.get('names', {}))}")
    print(f"  Classes: {list(data_yaml.get('names', {}).values())}")

## Step 4: Train YOLOv8 Model

Training parameters:
- **Model:** YOLOv8n (nano - faster training)
- **Epochs:** 100 (can increase for better accuracy)
- **Image size:** 640×640
- **Batch size:** 16 (auto-scales based on GPU memory)
- **Device:** GPU (Colab T4 or A100)

In [ ]:
from ultralytics import YOLO
import torch

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

# Load YOLOv8 nano model
print("Loading YOLOv8 nano model...")
model = YOLO('yolov8n.pt')

# Get dataset path
data_yaml_path = os.path.join(dataset_path, 'data.yaml')

# Train the model
print("Starting training...\n")
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,  # Early stopping if no improvement for 20 epochs
    device=0,  # GPU device ID (0 = first GPU)
    project='coral_detection',
    name='yolov8n_coral_v1',
    save=True,
    exist_ok=False,
    pretrained=True,
    optimizer='SGD',
    verbose=True,
    seed=42
)

print("\n✓ Training completed!")

## Step 5: Evaluate Model Performance

In [ ]:
# Validate model
print("Running validation...\n")
metrics = model.val()

print("\n" + "="*50)
print("VALIDATION METRICS")
print("="*50)
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print("="*50)

## Step 6: Test on Sample Images

In [ ]:
# Get test images from the dataset
# Roboflow structure: dataset_path/test/images/ OR dataset_path/images/test/
test_dir = os.path.join(dataset_path, 'test', 'images')
if not os.path.exists(test_dir):
    test_dir = os.path.join(dataset_path, 'images', 'test')

if not os.path.exists(test_dir):
    print(f"❌ Test directory not found at:")
    print(f"   {os.path.join(dataset_path, 'test', 'images')}")
    print(f"   {os.path.join(dataset_path, 'images', 'test')}")
    print(f"\nUsing validation images instead...")
    test_dir = os.path.join(dataset_path, 'valid', 'images')
    if not os.path.exists(test_dir):
        test_dir = os.path.join(dataset_path, 'images', 'valid')

test_images = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png'))][:5]

print(f"Testing on {len(test_images)} sample images...\n")

# Run inference
for img_path in test_images:
    print(f"\nTesting: {os.path.basename(img_path)}")
    results = model.predict(img_path, conf=0.25)
    
    # Display results
    if len(results[0]) > 0:
        detections = results[0]
        print(f"  Detections: {len(detections.boxes)} objects found")
        for i, box in enumerate(detections.boxes):
            cls_id = int(box.cls[0])
            cls_name = results[0].names[cls_id]
            conf = float(box.conf[0])
            print(f"    {i+1}. {cls_name}: {conf:.2%} confidence")
    else:
        print("  No detections found")

## Step 7: Save and Export Model

Save the trained model in multiple formats for deployment

In [ ]:
# Save model paths
model_dir = '/content/coral_detection/yolov8n_coral_v1'

# The best model is already saved at:
best_model_path = os.path.join(model_dir, 'weights', 'best.pt')
last_model_path = os.path.join(model_dir, 'weights', 'last.pt')

print("Model files:")
if os.path.exists(best_model_path):
    size_mb = os.path.getsize(best_model_path) / 1e6
    print(f"  ✓ Best model: {best_model_path} ({size_mb:.1f} MB)")

if os.path.exists(last_model_path):
    size_mb = os.path.getsize(last_model_path) / 1e6
    print(f"  ✓ Last model: {last_model_path} ({size_mb:.1f} MB)")

# Export to ONNX for deployment
print("\nExporting to ONNX format...")
try:
    model.export(format='onnx', imgsz=640)
    print("✓ ONNX export successful")
except Exception as e:
    print(f"ONNX export: {e}")

# Export to TorchScript
print("Exporting to TorchScript format...")
try:
    model.export(format='torchscript', imgsz=640)
    print("✓ TorchScript export successful")
except Exception as e:
    print(f"TorchScript export: {e}")

print("\n✓ All exports completed!")

## Step 8: Download Trained Model

Download the trained model to your local machine

In [ ]:
from google.colab import files

# Zip the trained model
!cd /content && zip -r coral_model_v1.zip coral_detection/ -q

# Download
print("Downloading trained model...")
files.download('/content/coral_model_v1.zip')
print("✓ Download started! Check your Downloads folder.")

## Training Complete! 🎉

Your YOLOv8 model has been trained on 341 augmented coral images with 7 classes.

**Next Steps:**
1. Download the trained model
2. Integrate into your Django application
3. Replace the mock AI generator with real YOLO inference
4. Test on new coral images

**Model Location:** `/content/coral_detection/yolov8n_coral_v1/weights/best.pt`

**Key Files:**
- `best.pt` - Best trained model
- `last.pt` - Last checkpoint
- `results.csv` - Training metrics
- `results.png` - Training plots